Group Project for Model visualization

Ricardo Guerrero

Akshith Kumar Dharavath

Poorna Sai Karthik Lallaboyina

# DataFitLab: Exploring Support Vector Machines

## 1. The Mathematical Structure of the Dataset
Before we can classify data, we must understand its shape. The `make_circles` function generates a 2D synthetic dataset that is explicitly **non-linearly separable.**
What does this mean?

When the code runs `X, y = make_circles(...)`, it is generating two mathematical structures:

1. **The Feature Matrix ($X$):** A matrix of shape $(300, 2)$, representing $300$ samples with $2$ features ($x$ and $y$ coordinates). 
$$X = \begin{bmatrix} x_1 & y_1 \\ x_2 & y_2 \\ \vdots & \vdots \\ x_{300} & y_{300} \end{bmatrix} \in \mathbb{R}^{300 \times 2}$$

2. **The Target Vector ($y$):** A 1D array of shape $(300, 1)$, containing binary labels representing the inner circle ($1$) and the outer circle ($0$).
$$\mathbf{y} = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_{300} \end{bmatrix} \in \{0, 1\}^{300}$$

Because the class $1$ data points are entirely surrounded by class $0$ data points, no single straight line (a 1D hyperplane) can separate them in $\mathbb{R}^2$ space.

CELL 1: GLOBAL Imports & Unified Dataset

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.model_selection import validation_curve
from ipywidgets import interact, FloatLogSlider
import warnings

# Suppress warnings for a clean presentation
warnings.filterwarnings('ignore')

# 1. Generate the Unified "Teaching" Dataset
# n_samples=300: Enough data for cross-validation
# noise=0.1: Messy enough to demonstrate overfitting
# factor=0.3: Clear separation for the 3D visualization
X, y = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

# Extract 2D coordinates for easy plotting
x_coords = X[:, 0]
y_coords = X[:, 1]

print(f"Global Dataset Generated: {X.shape[0]} samples.")

Global Dataset Generated: 300 samples.


## 2. The Kernel Trick: Dimensionality Expansion
To solve the non-linear problem, Support Vector Machines (SVM) utilize the **Kernel Trick**. By mapping our data into a higher-dimensional space using a mapping function $\phi(\mathbf{x})$. data that is entangled in lower dimensions can now become linearly separable.

In this demonstration, we manually apply a quadratic mapping function. We calculate a new feature, **$z$**, based on the squared distance from the origin:
$$z = x^2 + y^2$$

We then concatenate this to our original matrix to create our new 3D feature space:
$$X_{3D} = \begin{bmatrix} x_1 & y_1 & x_1^2 + y_1^2 \\ x_2 & y_2 & x_2^2 + y_2^2 \\ \vdots & \vdots & \vdots \end{bmatrix} \in \mathbb{R}^{300 \times 3}$$
##### Notice that the matrix is now larger because of that concatenation of the **$z$** feature substitution we made at the third column.

### **HOW the SVM calculates a specific Hyperplane**
Once the data is in 3D, the outer circle (which has larger $x$ and $y$ values) gets pushed much higher along the Z-axis than the inner circle. The `scikit-learn` optimizer now searches for a weight vector ($\mathbf{w}$) and a bias ($b$) to define a flat 2D plane that slices between them. 

The SVM optimization objective is to **maximize the margin** between the two classes. It solves for the hyperplane equation:
$$\mathbf{w}^T \mathbf{x} + b = 0 \implies w_0x + w_1y + w_2z + b = 0$$
##### Notice how the hyperplane equation contains 3 variables. However, we know the $z$ variable is dependent on the $x$ and $y$ variable by definition. That is where the shape comes from.

Because the separation of the classes is now almost entirely dependent on height, the SVM heavily weights the $z$ component. To visualize this in Plotly, we algebraically rearrange the SVM's output equation to solve for $z$, allowing us to plot the surface:
$$z = \frac{-(w_0x + w_1y + b)}{w_2}$$

CELL 2: The 3D Kernel Trick Visualization (Plotly)
the cell uses the unified 'X & y' variables. It calculates the Z-lift, trains the base model, and renders the interactive 2D-to-3D grid.

In [ ]:
# 1. The "Lifting" Logic (Manual Kernel Trick)
z_coords = (x_coords ** 2) + (y_coords ** 2)
X_3d = np.column_stack((x_coords, y_coords, z_coords))

# 2. Train the SVM to find the perfect 3D Hyperplane
model_3d = SVC(kernel='linear', C=1.0)
model_3d.fit(X_3d, y)

# Extract weights and support vectors
w0, w1, w2 = model_3d.coef_[0]
bias = model_3d.intercept_[0]
sv_coords_3d = model_3d.support_vectors_

# DEMONSTRATION: Print the Calculated Matrices
print("-" * 55)
print(" SVM HYPERPLANE MATRICES (3D LIFT)")
print("-" * 55)
print(f"Weight Vector (W): [{w0:.4f}, {w1:.4f}, {w2:.4f}]")
print(f"Bias (b):          {bias:.4f}")
print("-" * 55)


# If you plug any data point's X, Y, and Z coordinates into this exact equation,
# a positive result means it belongs to the inner circle,
# and a negative result means it belongs to the outer circle.
print(f"Margin Equation:   ({w0:.4f} * x) + ({w1:.4f} * y) + ({w2:.4f} * z) + {bias:.4f} = 0")
print(f"Support Vectors:   {len(sv_coords_3d)} points dictating the boundary")
print("-" * 55)

# 3. Create the Grid for the Hyperplane Surface
x_min, x_max = x_coords.min() - 0.2, x_coords.max() + 0.2
y_min, y_max = y_coords.min() - 0.2, y_coords.max() + 0.2
xx_3d, yy_3d = np.meshgrid(np.linspace(x_min, x_max, 30),
                           np.linspace(y_min, y_max, 30))
zz_3d = -(w0 * xx_3d + w1 * yy_3d + bias) / w2

# 4. Build the Plotly Figure
fig = go.Figure()

# Base Data
fig.add_trace(go.Scatter3d(
    x=x_coords, y=y_coords, z=z_coords, mode='markers',
    marker=dict(size=4, color=y, colorscale='Viridis', opacity=0.6),
    name='Data Points', hovertemplate='X: %{x:.2f}<br>Y: %{y:.2f}<br>Z: %{z:.2f}'
))

# Support Vectors
fig.add_trace(go.Scatter3d(
    x=sv_coords_3d[:, 0], y=sv_coords_3d[:, 1], z=sv_coords_3d[:, 2], mode='markers',
    marker=dict(size=8, color='rgba(0,0,0,0)', line=dict(color='red', width=3)),
    name='Support Vectors', hoverinfo='skip'
))

# Hyperplane
fig.add_trace(go.Surface(
    x=xx_3d, y=yy_3d, z=zz_3d, colorscale='Greys', opacity=0.5, showscale=False, name='Hyperplane'
))

# 5. Interactive Animation UI
zeros_base = np.zeros_like(z_coords)
zeros_sv = np.zeros_like(sv_coords_3d[:, 2])
zeros_plane = np.zeros_like(zz_3d)

updatemenus = [dict(
    type="buttons", direction="right", x=0.5, y=1.1, showactive=True,
    buttons=list([
        dict(label="1. View 2D Data", method="update",
             args=[{"z": [zeros_base, zeros_sv, zeros_plane], "visible": [True, True, False]}, 
                   {"scene.camera.eye": dict(x=0, y=0.01, z=2.5)}]),
        dict(label="2. View 3D Space", method="update",
             args=[{"z": [z_coords, sv_coords_3d[:, 2], zz_3d], "visible": [True, True, True]}, 
                   {"scene.camera.eye": dict(x=1.5, y=1.5, z=1.5)}])
    ])
)]

fig.update_layout(
    title='DataFitLab: The Kernel Trick (2D to 3D Lift)',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z (x² + y²)', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))),
    width=1200, height=700, margin=dict(l=0, r=0, b=0, t=40),
    updatemenus=updatemenus, transition=dict(duration=1200, easing="cubic-in-out")
)

fig.show()

-------------------------------------------------------
 SVM HYPERPLANE MATRICES (3D LIFT)
-------------------------------------------------------
Weight Vector (W): [-0.0779, -0.0015, -4.1966]
Bias (b):          1.9783
-------------------------------------------------------
Margin Equation:   (-0.0779 * x) + (-0.0015 * y) + (-4.1966 * z) + 1.9783 = 0
Support Vectors:   23 points dictating the boundary
-------------------------------------------------------


## 3. Model Complexity: The Radial Basis Function (RBF)
While manual lifting works for simple geometry, real-world data requires more robust transformations. Thus, the 3D kernel trick is not always effective. The default kernel for an SVM is the **Radial Basis Function (RBF)**. 

Instead of just adding a $z$ coordinate, the RBF kernel calculates the similarity between any two points $\mathbf{x}_i$ and $\mathbf{x}_j$ projecting them into a theoretically infinite-dimensional space:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$$

### **The Role of Gamma ($\gamma$) in Model Complexity**
The $\gamma$ parameter dictates the "spread" of the kernel. 
* **Low $\gamma$:** The exponential decays slowly. Each support vector has a wide area of influence, resulting in a smooth, rigid decision boundary (**High Bias / Underfitting**).
* **High $\gamma$:** The exponential decays rapidly. Each support vector only influences data points immediately adjacent to it, creating a highly fragmented boundary that perfectly wraps around individual points (**High Variance / Overfitting**).

### **The Cross-Validation Matrix**
To prove this tradeoff, the `validation_curve` function tests 30 different $\gamma$ values. For each value, it splits the dataset 5 times (K-Fold Cross-Validation). This generates a massive score matrix:
$$\text{Scores} = \begin{bmatrix} s_{1,1} & \dots & s_{1,5} \\ \vdots & \ddots & \vdots \\ s_{30,1} & \dots & s_{30,5} \end{bmatrix} \in \mathbb{R}^{30 \times 5}$$
By calculating the mean across the columns (`axis=1`), we extract the exact error rate for each $\gamma$ value, generating the classic U-shaped Validation Error curve shown in the dashboard below.

This cell takes the exact same data we recorded (X & y), and tests it across a large range of complexities (extra dimensions) and launches an interactive UI to test it

In [18]:
print("Pre-computing validation curves for the global dataset... please wait.")

# 1. Define range of Gammas to test and calculate Error Curves
gamma_range = np.logspace(-3, 2, 30)
train_scores, val_scores = validation_curve(
    SVC(kernel='rbf', C=1.0), X, y, param_name="gamma", 
    param_range=gamma_range, cv=5, scoring="accuracy", n_jobs=-1
)

# Convert accuracy to error
train_error_mean = 1.0 - np.mean(train_scores, axis=1)
val_error_mean = 1.0 - np.mean(val_scores, axis=1)

# 2. Define the Interactive Function
def interactive_dashboard(gamma_val):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- LEFT PANEL: Live Boundary ---
    live_model = SVC(kernel='rbf', C=1.0, gamma=gamma_val)
    live_model.fit(X, y)
    
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 100))
    Z = live_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax1.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax1.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', s=30)
    ax1.set_title(f"Decision Boundary (Gamma = {gamma_val:.3f})")
    ax1.set_xticks(())
    ax1.set_yticks(())
    
    # --- RIGHT PANEL: Tradeoff Curve ---
    ax2.plot(gamma_range, train_error_mean, label="Training Error (Bias)", color='blue', lw=2)
    ax2.plot(gamma_range, val_error_mean, label="Validation Error (Variance)", color='orange', lw=2)
    
    closest_idx = (np.abs(gamma_range - gamma_val)).argmin()
    
    ax2.axvline(x=gamma_val, color='gray', linestyle='--', alpha=0.7)
    ax2.scatter([gamma_val], [train_error_mean[closest_idx]], color='blue', s=100, zorder=5)
    ax2.scatter([gamma_val], [val_error_mean[closest_idx]], color='orange', s=100, zorder=5)
    
    ax2.set_xscale('log')
    ax2.set_title("Bias-Variance Tradeoff")
    ax2.set_xlabel("Model Complexity (Gamma)")
    ax2.set_ylabel("Error Rate")
    ax2.legend(loc="upper left")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 3. Launch the UI
slider = FloatLogSlider(
    value=0.1, base=10, min=-3, max=2, step=0.1, 
    description='Gamma (γ):', continuous_update=False
)

print("Ready! Adjust the slider below:")
interact(interactive_dashboard, gamma_val=slider);

Pre-computing validation curves for the global dataset... please wait.
Ready! Adjust the slider below:


interactive(children=(FloatLogSlider(value=0.1, continuous_update=False, description='Gamma (γ):', max=2.0, mi…